In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["numpy","pandas","pyreadr","pyarrow","rdata","great_tables","statsmodels", "pygam",
            "scikit-learn" ,"matplotlib" , "qiskit-aer" , "qiskit","qiskit-machine-learning","qkm_swap_test"]:
    try:
        install(pkg)
    except Exception as e:
        print(f"Cảnh báo: {pkg} — {e}")

print("Tất cả thư viện đã sẵn sàng.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH             = '/content/drive/MyDrive/...'   # path đến osf_mini.Rda
VAR_META              = '/content/drive/MyDrive/...'   # path đến osf_var_meta.xlsx
PATH_TO_SAVE_DATA_ALL = '/content/drive/MyDrive/...'   # path lưu dt_all

In [ ]:
import pandas as pd
import numpy as np
import pyreadr

def show_infor(df):
    # -----------------------------------------------------------------
    # BASIC INFORMATION :
    rows,cols = df.shape
    print(f"|{rows} rows | {cols} colums |")
    # -----------------------------------------------------------------
    # DATA TYPES :
    numeric_cols = df.select_dtypes(include=np.number).columns
    categorical_cols = df.select_dtypes(include="object").columns
    datetime_cols = df.select_dtypes(include="datetime").columns

    print(f"\nNumeric Features      : {len(numeric_cols)}")
    print(f"Categorical Features  : {len(categorical_cols)}")
    print(f"Datetime Features     : {len(datetime_cols)}")
    # -----------------------------------------------------------------
    # MISSING VALUE
    missing_total = df.isnull().sum().sum()
    total_cells = rows * cols
    missing_percent = (missing_total / total_cells) * 100

    print(f"\nMissing Values        : {missing_total}")
    print(f"Missing Percentage    : {missing_percent:.2f}%")
    # ----------------------------------------------------------------
    # UNIQUE VALUES
    if "PID" in df.columns:
        print(f"\nUnique Participants   : {df['PID'].nunique()}")

    if "study" in df.columns:
        print(f"Studies               : {df['study'].nunique()}")
        print(f"Study Labels          : {df['study'].unique()}")
    # -----------------------------------------------------------------
    # DUPLICATES
    duplicates = df.duplicated().sum()

    print(f"\nDuplicate Rows        : {duplicates}")
    # ----------------------------------------------------------------
    # MISSING VALUE BY COLUMN
    print("\n======== Missing Values by Column ========")

    missing_by_col = df.isnull().sum()

    for col, val in missing_by_col.items():
        percent = (val / rows) * 100
        print(f"{col:<40} : {val:<5} ({percent:.2f}%)")
        # ----------------------------------------------------------------
    # SHOW FIRST 5 ROWS
    print("========  Show first 5 Rows :  ==========")
    print(df.head(5))

result = pyreadr.read_r(DATA_PATH)
dt_raw = result[None]

var_meta = pd.read_excel(VAR_META)

print("-------------------------------------------------------")
print("Infor of OSF_MINI.Rda : ")
show_infor(dt_raw)
print("-------------------------------------------------------")
print("Infor of var_meta.xlsx : ")
show_infor(var_meta)
print("-------------------------------------------------------")

In [ ]:
import numpy as np
import pandas as pd


def clean_m(M: pd.DataFrame, c: float = 55) -> dict:
    """
    Clean up matrix based on mean completion rates.

    Removes rows and columns from a matrix whose mean completion rate
    (values multiplied by 100) falls below a specified threshold.

    Parameters
    ----------
    M : pd.DataFrame
        A numeric DataFrame. Rows and columns with names starting with "PP_"
        are considered for cleaning.
    c : float
        Threshold for mean completion rate in percentage (default is 55).

    Returns
    -------
    dict with keys:
        - reducedMatrix  : The cleaned DataFrame.
        - rowNamesIn     : Row names remaining in the cleaned matrix.
        - colNamesIn     : Column names remaining in the cleaned matrix.
        - rowNamesOut    : Row names removed from the matrix.
        - colNamesOut    : Column names removed from the matrix.
        - rowMeans       : Row mean completion rates of the original matrix.
        - colMeans       : Column mean completion rates of the original matrix.
    """
    M = M.copy()

    row_names_out = []
    col_names_out = []

    max_iter = len(M.index) + len(M.columns)

    for i in range(1, max_iter + 1):
        # Calculate row and column mean completion rates
        row_means = M.mean(axis=1) * 100   # Series indexed by row names
        col_means = M.mean(axis=0) * 100   # Series indexed by col names
        rc_means  = pd.concat([row_means, col_means])

        # Name with the lowest mean
        rm = rc_means.idxmin()
        rc = "row" if str(rm).startswith("PP_") else "col"

        if (rc_means >= c).all():
            print(
                f"{i}: All row- and column means are over {c}%. "
                f"The final matrix has {M.shape[0]} rows and {M.shape[1]} columns."
            )
            return {
                "reducedMatrix": M,
                "rowNamesIn":    list(M.index),
                "colNamesIn":    list(M.columns),
                "rowNamesOut":   row_names_out,
                "colNamesOut":   col_names_out,
                "rowMeans":      row_means,
                "colMeans":      col_means,
            }

        if rc == "row":
            print(
                f"{i}: Row {rm} had a completion rate of "
                f"{rc_means.min():.2f}% and was removed."
            )
            M = M.drop(index=rm)
            row_names_out.append(rm)

        else:
            print(
                f"{i}: Column {rm} had a completion rate of "
                f"{rc_means.min():.2f}% and was removed."
            )
            M = M.drop(columns=rm)
            col_names_out.append(rm)

    return None  # Should not be reached under normal circumstances

### **Study 01 :**

In [ ]:
import pandas as pd

dt_s1_availability = (
    dt_raw
    .query("study == 'S1'")
    .assign(present=1)
    .pivot_table(
        index="PID",
        columns="TIDnum",
        values="present",
        aggfunc="max",
        fill_value=0
    )
)

dt_s1_availability.index   = [f"PP_{int(i)}" for i in dt_s1_availability.index]
dt_s1_availability.columns = [f"t_{int(i)}"  for i in dt_s1_availability.columns]

dt_s1_red_info = clean_m(M=dt_s1_availability, c=55)

In [ ]:
import pandas as pd
import numpy as np

# R: PIDout <- gsub("PP_", "", dtS1RedInfo$rowNamesOut) %>% as.numeric
pid_out = [int(x.replace("PP_", "")) for x in dt_s1_red_info["rowNamesOut"]]

# R: TIDout <- gsub("t_", "", dtS1RedInfo$colNamesOut) %>% as.numeric
tid_out = [int(x.replace("t_", "")) for x in dt_s1_red_info["colNamesOut"]]

# R: TIDInRed <- gsub("t_", "", dtS1RedInfo$colNamesIn) %>% as.numeric
tid_in_red = [int(x.replace("t_", "")) for x in dt_s1_red_info["colNamesIn"]]

# R: TIDIn <- seq(min(TIDInRed), max(TIDInRed), 1)
tid_in = list(range(min(tid_in_red), max(tid_in_red) + 1))

# R: dtS1Red <- dt_raw %>% filter(study == "S1") %>% filter(!PID %in% PIDout, TIDnum %in% TIDIn) %>% mutate(TIDnum = TIDnum - min(TIDnum))
dt_s1_red = (
    dt_raw
    .query("study == 'S1'")
    .loc[lambda df: ~df["PID"].isin(pid_out) & df["TIDnum"].isin(tid_in)]
    .assign(TIDnum=lambda df: df["TIDnum"] - df["TIDnum"].min())
    .copy()
)

# R: rm(PIDout, TIDout, TIDInRed, TIDIn)
del pid_out, tid_out, tid_in_red, tid_in

# R: fix glitch where survey was sent too early
mask = (dt_s1_red["PID"] == 7) & (dt_s1_red["created"].astype(str) == "2018-05-18 11:46:40")
dt_s1_red.loc[mask, "TIDnum"] = 10
dt_s1_red.loc[mask, "TID"]    = "2018-05-18 Morning"

# ---------------------------------------------------------------------------
# Styled HTML table  (R: cell_spec + kbl + kable_classic + scroll_box)
# ---------------------------------------------------------------------------
dt_s1_availability_red_kbl = dt_s1_red_info["reducedMatrix"].copy()

def style_cell(val):
    if val == 1:
        return (
            "background-color: #1D9E75;"   # teal đậm
            "color: #E1F5EE;"              # teal nhạt
            "border-radius: 6px;"
            "font-weight: 500;"
        )
    else:
        return (
            "background-color: #D85A30;"   # coral đậm
            "color: #FAECE7;"              # coral nhạt
            "border-radius: 6px;"
            "font-weight: 500;"
        )

styled = (
    dt_s1_availability_red_kbl
    .style
    .applymap(style_cell)
    .set_caption("Study 1: Final Data Availability")
    .set_table_styles([
        {"selector": "table",
         "props": [("font-family", "Cambria"),
                   ("border-collapse", "separate"),
                   ("border-spacing", "100px")]},      # tạo khoảng cách giữa các cell
        {"selector": "th, td",
         "props": [("text-align", "center"),
                   ("padding", "6px 10px"),
                   ("border-radius", "6px")]},
        {"selector": "tr:hover td",
         "props": [("opacity", "0.85")]},
    ])
)


# Render inside a scrollable box and display (Jupyter / IPython)
from IPython.display import HTML

scrollable_html = f"""
<div style="width:100%; height:500px; overflow:auto;
            border-radius:10px; border:1px solid #e0e0e0; padding:8px;">
  {styled.to_html()}
</div>
"""

HTML(scrollable_html)

### **Study 02:**

In [ ]:
import pandas as pd

dt_s2_availability = (
    dt_raw
    .query("study == 'S2'")
    .assign(present=1)
    .pivot_table(
        index="PID",
        columns="TIDnum",
        values="present",
        aggfunc="max",
        fill_value=0
    )
)

dt_s2_availability.index   = [f"PP_{int(i)}" for i in dt_s2_availability.index]
dt_s2_availability.columns = [f"t_{int(i)}"  for i in dt_s2_availability.columns]

dt_s2_red_info = clean_m(M=dt_s2_availability, c=55)

In [ ]:
# --- Strip prefixes and build TIDIn range ---
pid_out    = [int(x.replace("PP_", "")) for x in dt_s2_red_info["rowNamesOut"]]
tid_out    = [int(x.replace("t_",  "")) for x in dt_s2_red_info["colNamesOut"]]
tid_in_red = [int(x.replace("t_",  "")) for x in dt_s2_red_info["colNamesIn"]]
tid_in     = list(range(min(tid_in_red), max(tid_in_red) + 1))

# --- Filter dt_raw for S2 ---
dt_s2_red = (
    dt_raw
    .query("study == 'S2'")
    .loc[lambda df: ~df["PID"].isin(pid_out) & df["TIDnum"].isin(tid_in)]
    .assign(TIDnum=lambda df: df["TIDnum"] - df["TIDnum"].min())
    .copy()
)

del pid_out, tid_out, tid_in_red, tid_in

# --- Remove glitch entries (filter by PID + TIDnum + ended) ---
glitch_mask = (
    (dt_s2_red["PID"] == 46) & (dt_s2_red["TIDnum"] == 57) & (dt_s2_red["ended"].astype(str) == "2018-12-20 21:58:31")
) | (
    (dt_s2_red["PID"] == 46) & (dt_s2_red["TIDnum"] == 59) & (dt_s2_red["ended"].astype(str) == "2018-12-20 21:58:31")
)
dt_s2_red = dt_s2_red[~glitch_mask].copy()

# --- Styled availability table ---
dt_s2_availability_red_kbl = dt_s2_red_info["reducedMatrix"].copy()

def style_cell(val):
    if val == 1:
        return "background-color: #1D9E75; color: #E1F5EE; border-radius: 6px; font-weight: 500;"
    return     "background-color: #D85A30; color: #FAECE7; border-radius: 6px; font-weight: 500;"

styled = (
    dt_s2_availability_red_kbl
    .style
    .applymap(style_cell)
    .set_caption("Study 2: Final Data Availability")
    .set_table_styles([
        {"selector": "table",
         "props": [("font-family", "Cambria"),
                   ("border-collapse", "separate"),
                   ("border-spacing", "3px")]},
        {"selector": "th, td",
         "props": [("text-align", "center"),
                   ("padding", "6px 10px"),
                   ("border-radius", "6px")]},
        {"selector": "tr:hover td",
         "props": [("opacity", "0.85")]},
    ])
)

scrollable_html = f"""
<div style="width:100%; height:500px; overflow:auto;
            border-radius:10px; border:1px solid #e0e0e0; padding:8px;">
  {styled.to_html()}
</div>
"""

HTML(scrollable_html)

### **Study 03:**

In [ ]:
import pandas as pd

dt_s3_availability = (
    dt_raw
    .query("study == 'S3'")
    .assign(present=1)
    .pivot_table(
        index="PID",
        columns="TIDnum",
        values="present",
        aggfunc="max",
        fill_value=0
    )
)

dt_s3_availability.index   = [f"PP_{int(i)}" for i in dt_s3_availability.index]
dt_s3_availability.columns = [f"t_{int(i)}"  for i in dt_s3_availability.columns]

dt_s3_red_info = clean_m(M=dt_s3_availability, c=55)

In [ ]:
# --- Strip prefixes and build TIDIn range ---
pid_out    = [int(x.replace("PP_", "")) for x in dt_s3_red_info["rowNamesOut"]]
tid_out    = [int(x.replace("t_",  "")) for x in dt_s3_red_info["colNamesOut"]]
tid_in_red = [int(x.replace("t_",  "")) for x in dt_s3_red_info["colNamesIn"]]
tid_in     = list(range(min(tid_in_red), max(tid_in_red) + 1))

# --- Filter dt_raw for S3 ---
dt_s3_red = (
    dt_raw
    .query("study == 'S3'")
    .loc[lambda df: ~df["PID"].isin(pid_out) & df["TIDnum"].isin(tid_in)]
    .assign(TIDnum=lambda df: df["TIDnum"] - df["TIDnum"].min())
    .copy()
)

del pid_out, tid_out, tid_in_red, tid_in

# --- Styled availability table ---
dt_s3_availability_red_kbl = dt_s3_red_info["reducedMatrix"].copy()

styled = (
    dt_s3_availability_red_kbl
    .style
    .applymap(style_cell)                          # reuse style_cell từ S1/S2
    .set_caption("Study 3: Final Data Availability")
    .set_table_styles([
        {"selector": "table",
         "props": [("font-family", "Cambria"),
                   ("border-collapse", "separate"),
                   ("border-spacing", "3px")]},
        {"selector": "th, td",
         "props": [("text-align", "center"),
                   ("padding", "6px 10px"),
                   ("border-radius", "6px")]},
        {"selector": "tr:hover td",
         "props": [("opacity", "0.85")]},
    ])
)

scrollable_html = f"""
<div style="width:100%; height:500px; overflow:auto;
            border-radius:10px; border:1px solid #e0e0e0; padding:8px;">
  {styled.to_html()}
</div>
"""

HTML(scrollable_html)

# **S1 + S2 + S3 -> dtAll**

In [ ]:
import pandas as pd


def miss_info(full: pd.DataFrame, reduced: pd.DataFrame) -> pd.DataFrame:
    """
    Extract missing information from datasets.

    Compares the full and reduced datasets and returns a summary table
    of the missing data.

    Parameters
    ----------
    full    : pd.DataFrame — the full dataset (must have PID, TID columns)
    reduced : pd.DataFrame — the reduced dataset (must have PID, TID columns)

    Returns
    -------
    pd.DataFrame with columns:
        nFull, nRed, nDif, nDifPerc,
        pptFull, pptRed, pptDif, pptDifPerc,
        timeFull, timeRed, timeDif, timeDifPerc
    """
    n_full    = len(full)
    n_red     = len(reduced)
    ppt_full  = full["PID"].nunique()
    ppt_red   = reduced["PID"].nunique()
    time_full = full["TID"].nunique()
    time_red  = reduced["TID"].nunique()

    miss_tab = pd.DataFrame([{
        "nFull":       n_full,
        "nRed":        n_red,
        "nDif":        n_full - n_red,
        "nDifPerc":    (n_full - n_red) / n_full * 100,

        "pptFull":     ppt_full,
        "pptRed":      ppt_red,
        "pptDif":      ppt_full - ppt_red,
        "pptDifPerc":  (ppt_full - ppt_red) / ppt_full * 100,

        "timeFull":    time_full,
        "timeRed":     time_red,
        "timeDif":     time_full - time_red,
        "timeDifPerc": (time_full - time_red) / time_full * 100,
    }])

    return miss_tab

In [ ]:
import pandas as pd
from IPython.display import HTML

# --- Compute missingness per study ---
miss_s1 = miss_info(full=dt_raw.query("study == 'S1'"), reduced=dt_s1_red)
miss_s2 = miss_info(full=dt_raw.query("study == 'S2'"), reduced=dt_s2_red)
miss_s3 = miss_info(full=dt_raw.query("study == 'S3'"), reduced=dt_s3_red)

# --- Stack and add study column ---
miss_s123 = (
    pd.concat([miss_s1, miss_s2, miss_s3], ignore_index=True)
    .assign(study=[1, 2, 3])
    .loc[:, ["study", "nFull", "nRed", "nDif", "nDifPerc",
                       "pptFull", "pptRed", "pptDif", "pptDifPerc",
                       "timeFull", "timeRed", "timeDif", "timeDifPerc"]]
)

# --- Styled HTML table with grouped headers ---
col_names = ["Study"] + ["Full", "Reduced", "Δ", "%"] * 3

header_html = """
<tr>
  <th rowspan="2" style="{th}">Study</th>
  <th colspan="4" style="{th}">Measurements</th>
  <th colspan="4" style="{th}">Participants</th>
  <th colspan="4" style="{th}">Timepoints</th>
</tr>
<tr>
  {sub_headers}
</tr>
""".format(
    th="text-align:center; font-family:Cambria; padding:6px 10px; "
       "border-bottom: 1.5px solid #ccc;",
    sub_headers="".join(
        f'<th style="text-align:center; font-family:Cambria; padding:6px 8px;">{h}</th>'
        for h in ["Full", "Reduced", "Δ", "%"] * 3
    )
)

def fmt(val, col):
    """Round % columns to 2 dp, integers for counts."""
    if "Perc" in col:
        return f"{val:.2f}"
    return str(int(val))

rows_html = ""
for _, row in miss_s123.iterrows():
    cells = f'<td style="text-align:center; padding:6px 10px;">{int(row["study"])}</td>'
    for col in ["nFull","nRed","nDif","nDifPerc",
                "pptFull","pptRed","pptDif","pptDifPerc",
                "timeFull","timeRed","timeDif","timeDifPerc"]:
        cells += f'<td style="text-align:center; padding:6px 10px;">{fmt(row[col], col)}</td>'
    rows_html += f"<tr>{cells}</tr>"

table_html = f"""
<div style="font-family:Cambria;">
<table style="border-collapse:collapse; width:auto;">
  <caption style="font-weight:500; margin-bottom:8px;">Missingness Info by Study</caption>
  <thead style="border-top:1.5px solid #ccc;">
    {header_html}
  </thead>
  <tbody>
    {rows_html}
  </tbody>
</table>
</div>
"""

HTML(table_html)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import warnings


def MlCorMat(
    data,
    id,
    selection,
    labels=None,
    method="pearson",
    removeTriangle="upper",
    result="none"
):
    """
    Calculate Multilevel Correlation Matrix (Python port of R MlCorMat).

    Parameters
    ----------
    data        : pd.DataFrame
    id          : str – name of the cluster/participant ID column
    selection   : list[str] – variable names to include
    labels      : list[str] – display labels (defaults to selection)
    method      : "pearson" or "spearman"
    removeTriangle : "upper" or "lower" (unused in output table, kept for API parity)
    result      : "none" | "html" | "latex" (unused here, kept for API parity)

    Returns
    -------
    pd.DataFrame  – correlation matrix + descriptives, mirroring the R output
                    Upper triangle : between-person correlations
                    Lower triangle : within-person correlations
    """
    if labels is None:
        labels = selection

    n_vars = len(selection)

    # ── 1. Trait scores (participant means) ──────────────────────────────────
    df = data[[id] + selection].copy()
    trait_cols = {v: f"{v}_trait" for v in selection}

    group_means = df.groupby(id)[selection].transform("mean")
    for v in selection:
        df[f"{v}_trait"] = group_means[v]

    # ── 2. State scores (within-person deviations) ────────────────────────────
    for v in selection:
        df[f"{v}_state"] = df[v] - df[f"{v}_trait"]

    # ── 3. Within-person correlations (state scores) ─────────────────────────
    state_cols = [f"{v}_state" for v in selection]
    state_df   = df[state_cols].dropna()

    if method == "spearman":
        rWit_mat, pWit_mat = _spearman_matrix(state_df)
    else:
        rWit_mat, pWit_mat = _pearson_matrix(state_df)

    # ── 4. Between-person correlations (trait scores) ────────────────────────
    trait_df = df[[id] + [f"{v}_trait" for v in selection]].drop_duplicates(subset=id)
    trait_df = trait_df[[f"{v}_trait" for v in selection]].dropna()

    if method == "spearman":
        rBtw_mat, pBtw_mat = _spearman_matrix(trait_df)
    else:
        rBtw_mat, pBtw_mat = _pearson_matrix(trait_df)

    # ── 5. Format r + significance stars ─────────────────────────────────────
    def star(p):
        if   p < 0.001: return "***"
        elif p < 0.01:  return "** "
        elif p < 0.05:  return "*  "
        else:           return "   "

    def fmt_r(r, p):
        return f"{r:+.2f}{star(p)}"

    # ── 6. Combined matrix (upper=between, lower=within) ─────────────────────
    rMlComb = np.full((n_vars, n_vars), "", dtype=object)
    for i in range(n_vars):
        for j in range(n_vars):
            if i == j:
                rMlComb[i, j] = "—"
            elif i < j:   # upper triangle → between
                rMlComb[i, j] = fmt_r(rBtw_mat[i, j], pBtw_mat[i, j])
            else:          # lower triangle → within
                rMlComb[i, j] = fmt_r(rWit_mat[j, i], pWit_mat[j, i])

    cor_df = pd.DataFrame(rMlComb, index=labels, columns=labels)

    # ── 7. Descriptives ───────────────────────────────────────────────────────
    trait_unique = (
        df[[id] + [f"{v}_trait" for v in selection]]
        .drop_duplicates(subset=id)
    )

    grand_means = trait_unique[[f"{v}_trait" for v in selection]].mean()
    btw_sd      = trait_unique[[f"{v}_trait" for v in selection]].std()

    # Within-SD: sqrt(mean of within-person variances)
    n_persons = df[id].nunique()
    within_sd = []
    for v in selection:
        var_within = (
            df[[id, v]].dropna()
            .groupby(id)[v]
            .var()          # within-person variance per person
        )
        within_sd.append(np.sqrt(var_within.sum() / n_persons))

    # ICC(1): between-var / total-var  (one-way random effects ANOVA)
    icc1_list, icc2_list = [], []
    for v in selection:
        sub = df[[id, v]].dropna()
        icc1, icc2 = _compute_icc(sub, id, v)
        icc1_list.append(icc1)
        icc2_list.append(icc2)

    desc_df = pd.DataFrame({
        "Grand Mean":  [f"{x:.2f}" for x in grand_means.values],
        "Between SD":  [f"{x:.2f}" for x in btw_sd.values],
        "Within SD":   [f"{x:.2f}" for x in within_sd],
        "ICC(1)":      [f"{x:.2f}" for x in icc1_list],
        "ICC(2)":      [f"{x:.2f}" for x in icc2_list],
    }, index=labels).T   # transpose so rows=stats, cols=variables

    # ── 8. Stack & return ─────────────────────────────────────────────────────
    print("Upper Triangle: Between participants; Lower Triangle: Within participants")
    out = pd.concat([cor_df, desc_df])
    return out


# ── Helpers ───────────────────────────────────────────────────────────────────

def _pearson_matrix(df):
    cols = df.columns.tolist()
    n = len(cols)
    r_mat = np.eye(n)
    p_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            mask = df[cols[i]].notna() & df[cols[j]].notna()
            r, p = stats.pearsonr(df.loc[mask, cols[i]], df.loc[mask, cols[j]])
            r_mat[i, j] = r_mat[j, i] = r
            p_mat[i, j] = p_mat[j, i] = p
    return r_mat, p_mat


def _spearman_matrix(df):
    cols = df.columns.tolist()
    n = len(cols)
    r_mat = np.eye(n)
    p_mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            mask = df[cols[i]].notna() & df[cols[j]].notna()
            r, p = stats.spearmanr(df.loc[mask, cols[i]], df.loc[mask, cols[j]])
            r_mat[i, j] = r_mat[j, i] = r
            p_mat[i, j] = p_mat[j, i] = p
    return r_mat, p_mat


def _compute_icc(df, id_col, val_col):
    """
    One-way random-effects ANOVA ICC.
    ICC(1) = (MSb - MSw) / (MSb + (k-1)*MSw)
    ICC(2) = (MSb - MSw) / MSb
    where k = average group size.
    """
    groups  = df.groupby(id_col)[val_col]
    n_total = len(df)
    n_groups = df[id_col].nunique()
    grand_mean = df[val_col].mean()

    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2
                     for _, g in groups)
    ss_within  = sum(((g - g.mean()) ** 2).sum() for _, g in groups)

    df_between = n_groups - 1
    df_within  = n_total  - n_groups

    ms_between = ss_between / df_between if df_between > 0 else np.nan
    ms_within  = ss_within  / df_within  if df_within  > 0 else np.nan

    k = (n_total - sum(len(g) ** 2 / n_total
                       for _, g in df.groupby(id_col)[val_col])) / (n_groups - 1)

    icc1 = (ms_between - ms_within) / (ms_between + (k - 1) * ms_within)
    icc2 = (ms_between - ms_within) / ms_between

    return float(np.clip(icc1, -1, 1)), float(np.clip(icc2, -1, 1))

In [ ]:
import pandas as pd
import numpy as np

# ── 1. Stack the three studies ───────────────────────────────────────────────
dtAll = pd.concat([
    dt_s1_red.drop(columns=["created", "ended"])
           .assign(study="S1")
           .pipe(lambda df: df.assign(**{
               col: pd.to_numeric(df[col], errors="coerce")
               for col in df.columns
               if col not in ("TID", "study")
           })),
    dt_s2_red.drop(columns=["created", "ended"]).assign(study="S2"),
    dt_s3_red.drop(columns=["created", "ended"]).assign(study="S3"),
], ignore_index=True)

# ── 2. Group ID (equivalent to cur_group_id) ─────────────────────────────────
dtAll["ID"] = dtAll.groupby(["study", "PID"]).ngroup() + 1

# ── 3. Date / week columns ───────────────────────────────────────────────────
dtAll["date"] = pd.to_datetime(
    dtAll["TID"].str.replace(r" .*", "", regex=True)
).dt.date

dtAll["week"] = pd.to_datetime(dtAll["date"]).dt.strftime("%G-W%V")

# ── 4. Reorder columns ───────────────────────────────────────────────────────
priority_cols = ["ID", "PID", "study", "date", "week", "TIDnum"]
remaining_cols = [c for c in dtAll.columns if c not in priority_cols]
dtAll = dtAll[priority_cols + remaining_cols]

# ── 5. Sort ──────────────────────────────────────────────────────────────────
dtAll = dtAll.sort_values(["ID", "TIDnum"]).reset_index(drop=True)

# ── 6. Drop columns that are ALL NA (keep if at least 2 non-NA values) ───────
dtAll = dtAll.loc[:, dtAll.notna().sum() > 1]

# ── 7. MlCorMat (equivalent of source + call) ────────────────────────────────
# from scripts.MlCorMat import MlCorMat   # adjust import path as needed

cluster1_vars   = var_meta.loc[var_meta["cluster"] == 1, "name"].tolist()
cluster1_labels = var_meta.loc[var_meta["cluster"] == 1, "label"].tolist()

allMlCor = MlCorMat(
    data      = dtAll,
    id        = "ID",
    selection = cluster1_vars,
    labels    = cluster1_labels,
)

# save dtAll
dtAll.to_csv(PATH_TO_SAVE_DATA_ALL, index=False)

In [ ]:
from great_tables import GT, loc, style

col_names = [
    "Int: Accidental", "Int: Voluntary", "Int: Cooperative",
    "Int: Representative", "Int: Meaningful", "Int: Quality",
    "Int: Need Fulfil.", "Int: Need Fulfil. Partner", "Int: Attitude Partner",
    "Daytime Core Need", "Outgroup Attitude", "Well-being"
]

n_vars = 12

# ── Chuẩn bị DataFrame ────────────────────────────────────────────────────────
df = allMlCor.copy()
df.columns = col_names
df = df.reset_index().rename(columns={"index": "Variable"})

# Thêm cột Group để phân nhóm hàng
df.insert(1, "Group",
    ["Correlations"] * n_vars + ["Descriptives"] * (len(df) - n_vars)
)

# ── Tạo bảng ──────────────────────────────────────────────────────────────────
(
    GT(df, rowname_col="Variable", groupname_col="Group")
    .tab_header(title="Correlation Table and Descriptive Statistics")
    .cols_align(align="center")
    .cols_hide(columns="Group")   # ẩn cột Group khỏi body
    .tab_style(
        style=style.text(font="Cambria"),
        locations=loc.body()
    )
    .tab_style(
        style=[style.text(weight="bold"), style.fill(color="#e8e8e8")],
        locations=loc.row_groups()
    )
    .tab_source_note('"Int." = interaction.')
    .tab_source_note("Upper triangle: Between-person correlations; Lower triangle: Within-person correlations.")
    .tab_source_note("*** p < .001, ** p < .01, * p < .05")
    .tab_options(
        table_font_names=["Cambria", "serif"],
        table_width="auto",
        column_labels_font_weight="bold",
    )
)